# Kaggle PCB Data Cleaning & SAM Point Extraction

Cleans bounding box labels, maps classes to KiCad conventions, runs SAM, and exports coordinates to CSV and Excel.

In [ ]:
# Dependencies
!pip install -q opencv-python numpy pandas openpyxl matplotlib
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

import os
import json
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from segment_anything import sam_model_registry, SamPredictor

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# Model checkpoint
import urllib.request
SAM_CHECKPOINT = Path('sam_vit_b.pth')
if not SAM_CHECKPOINT.exists():
    urllib.request.urlretrieve('https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth', str(SAM_CHECKPOINT))

sam = sam_model_registry['vit_b'](checkpoint=str(SAM_CHECKPOINT))
sam.to(device=device)
sam.eval()
predictor = SamPredictor(sam)

In [ ]:
CLASS_MAP = {0: 'Cap1', 1: 'Cap2', 2: 'Cap3', 3: 'Cap4', 4: 'MOSFET', 5: 'Mov', 6: 'Resistor', 7: 'Transformer'}
PREFIX_MAP = {'Cap1': 'C', 'Cap2': 'C', 'Cap3': 'C', 'Cap4': 'C', 'MOSFET': 'Q', 'Mov': 'D', 'Resistor': 'R', 'Transformer': 'T'}
KICAD_FOOTPRINTS = {
    'Resistor': 'Resistor_SMD:R_0805_2012Metric',
    'Cap1': 'Capacitor_SMD:C_0805_2012Metric',
    'Cap2': 'Capacitor_SMD:C_1206_3216Metric',
    'Cap3': 'Capacitor_THT:CP_Radial_D6.3mm_P2.50mm',
    'Cap4': 'Capacitor_THT:CP_Radial_D8.0mm_P3.50mm',
    'MOSFET': 'Package_TO_SOT_SMD:SOT-23',
    'Mov': 'Diode_SMD:D_SOD-123',
    'Transformer': 'Transformer_SMD:Transformer_Bourns_SRF0703'
}

def clean_box(x1, y1, x2, y2, img_w, img_h, min_size=8):
    if x1 > x2: x1, x2 = x2, x1
    if y1 > y2: y1, y2 = y2, y1
    x1 = max(0, min(int(round(x1)), img_w - 1))
    y1 = max(0, min(int(round(y1)), img_h - 1))
    x2 = max(0, min(int(round(x2)), img_w))
    y2 = max(0, min(int(round(y2)), img_h))
    if (x2 - x1) < min_size or (y2 - y1) < min_size:
        return False, []
    return True, [x1, y1, x2, y2]

def extract_polygon_points(mask: np.ndarray, min_area: float = 15.0):
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours: return []
    cnt = max(contours, key=cv2.contourArea)
    if cv2.contourArea(cnt) < min_area: return []
    approx = cv2.approxPolyDP(cnt, 0.005 * cv2.arcLength(cnt, True), True)
    return [[round(float(p[0][0]), 2), round(float(p[0][1]), 2)] for p in approx]

In [ ]:
image_path = Path('dataset_split/train/images/VID20210601143927-96_jpg.rf.36de73b8200ee94d0bd4679407c9cd40.jpg')
labels_path = Path('dataset_split/train/labels/VID20210601143927-96_jpg.rf.36de73b8200ee94d0bd4679407c9cd40.txt')
output_dir = Path('kaggle_sam_output')
output_dir.mkdir(parents=True, exist_ok=True)

img = cv2.imread(str(image_path))
h, w = img.shape[:2]
predictor.set_image(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

boxes = []
with open(labels_path, 'r') as f:
    for line in f:
        p = line.strip().split()
        if len(p) >= 5:
            cid = int(float(p[0]))
            xc, yc, bw, bh = map(float, p[1:5])
            ok, box = clean_box((xc - bw/2)*w, (yc - bh/2)*h, (xc + bw/2)*w, (yc + bh/2)*h, w, h)
            if ok: boxes.append((box, cid))

records, shapes, ref_counts = [], [], {}
for idx, (b, cid) in enumerate(boxes, 1):
    masks, scores, _ = predictor.predict(box=np.array(b)[None, :], multimask_output=False)
    pts = extract_polygon_points(masks[0])
    if not pts:
        pts = [[float(b[0]), float(b[1])], [float(b[2]), float(b[1])], [float(b[2]), float(b[3])], [float(b[0]), float(b[3])]]
    cname = CLASS_MAP.get(cid, f'Class_{cid}')
    pfx = PREFIX_MAP.get(cname, 'U')
    ref_counts[pfx] = ref_counts.get(pfx, 0) + 1
    ref_des = f'{pfx}{ref_counts[pfx]}'
    records.append({
        'image_name': image_path.name, 'instance_id': idx, 'ref_des': ref_des, 'class_name': cname,
        'kicad_footprint': KICAD_FOOTPRINTS.get(cname, ''), 'confidence': round(float(scores[0]), 3),
        'num_points': len(pts), 'points_compact': '; '.join([f'({pt[0]},{pt[1]})' for pt in pts]),
        'points_json': json.dumps(pts)
    })

df = pd.DataFrame(records)
df.to_excel(output_dir / 'kaggle_sam_points.xlsx', index=False)
df.to_csv(output_dir / 'kaggle_sam_points.csv', index=False)
print(f'Done. {len(records)} components saved.')
df.head()